In [2]:
from azure.identity import DefaultAzureCredential
from azure.ai.ml import Input, MLClient
from azure.ai.ml.constants import (
    MonitorDatasetContext,
)
from azure.ai.ml.entities import (
    AlertNotification,
    BaselineDataRange,
    DataDriftSignal,
    DataQualitySignal,
    PredictionDriftSignal,
    DataDriftMetricThreshold,
    DataQualityMetricThreshold,
    FeatureAttributionDriftMetricThreshold,
    FeatureAttributionDriftSignal,
    PredictionDriftMetricThreshold,
    NumericalDriftMetrics,
    CategoricalDriftMetrics,
    DataQualityMetricsNumerical,
    DataQualityMetricsCategorical,
    MonitorFeatureFilter,
    MonitoringTarget,
    MonitorDefinition,
    MonitorSchedule,
    RecurrencePattern,
    RecurrenceTrigger,
    ServerlessSparkCompute,
    ReferenceData,
    ProductionData
)

In [3]:
# Get a handle to the workspace.
ml_client = MLClient.from_config(
    DefaultAzureCredential()
)

Found the config file in: .\config.json


In [ ]:
# Create a compute instance.
spark_compute = ServerlessSparkCompute(
    instance_type="standard_e4s_v3",
    runtime_version="3.4"
)

# Specify the online deployment if you have one.
monitoring_target = MonitoringTarget(
    ml_task="classification",
    endpoint_deployment_id="azureml:<endpoint-name>:<deployment-name>"
)

# Specify a look-back window size and offset to use. Omit this line to use the default values, which are listed in the documentation.
data_window = BaselineDataRange(lookback_window_size="P1D", lookback_window_offset="P0D")

# Set up the production data.
production_data = ProductionData(
    input_data=Input(
        type="uri_folder",
        path="azureml:<production-data-asset-name>:1"
    ),
    data_window=data_window,
    data_context=MonitorDatasetContext.MODEL_INPUTS,
)

# Set up the training data to use as a reference data asset.
reference_data_training = ReferenceData(
    input_data=Input(
        type="mltable",
        path="azureml:<reference-data-asset-name>:1"
    ),
    data_column_names={
        "target_column":"<target-column>"
    },
    data_context=MonitorDatasetContext.TRAINING,
)

# Create an advanced data drift signal.
features = MonitorFeatureFilter(top_n_feature_importance=10)

metric_thresholds = DataDriftMetricThreshold(
    numerical=NumericalDriftMetrics(
        jensen_shannon_distance=0.01
    ),
    categorical=CategoricalDriftMetrics(
        pearsons_chi_squared_test=0.02
    )
)

advanced_data_drift = DataDriftSignal(
    reference_data=reference_data_training,
    features=features,
    metric_thresholds=metric_thresholds,
    alert_enabled=True
)

# Create an advanced prediction drift signal.
metric_thresholds = PredictionDriftMetricThreshold(
    categorical=CategoricalDriftMetrics(
        jensen_shannon_distance=0.01
    )
)

advanced_prediction_drift = PredictionDriftSignal(
    reference_data=reference_data_training,
    metric_thresholds=metric_thresholds,
    alert_enabled=True
)

# Create an advanced data quality signal.
features = ['<feature-1>', '<feature-2>', '<feature-3>']

metric_thresholds = DataQualityMetricThreshold(
    numerical=DataQualityMetricsNumerical(
        null_value_rate=0.01
    ),
    categorical=DataQualityMetricsCategorical(
        out_of_bounds_rate=0.02
    )
)

advanced_data_quality = DataQualitySignal(
    reference_data=reference_data_training,
    features=features,
    metric_thresholds=metric_thresholds,
    alert_enabled=True
)

# Create a feature attribution drift signal.
metric_thresholds = FeatureAttributionDriftMetricThreshold(normalized_discounted_cumulative_gain=0.9)

feature_attribution_drift = FeatureAttributionDriftSignal(
    reference_data=reference_data_training,
    metric_thresholds=metric_thresholds,
    alert_enabled=True
)

# Put all monitoring signals in a dictionary.
monitoring_signals = {
    'data_drift_advanced':advanced_data_drift,
    'data_quality_advanced':advanced_data_quality,
    'feature_attribution_drift':feature_attribution_drift,
}

# Create an alert notification object.
alert_notification = AlertNotification(
    emails=['<email-address-1>', '<email-address-2>']
)

# Create the monitor definition.
monitor_definition = MonitorDefinition(
    compute=spark_compute,
    monitoring_target=monitoring_target,
    monitoring_signals=monitoring_signals,
    alert_notification=alert_notification
)

# Specify the schedule frequency.
recurrence_trigger = RecurrenceTrigger(
    frequency="<frequency-unit>",
    interval=<interval>,
    schedule=RecurrencePattern(hours=<start-hour>, minutes=<start-minutes>)
)

# Create the monitoring schedule.
model_monitor = MonitorSchedule(
    name="credit_default_monitor_advanced",
    trigger=recurrence_trigger,
    create_monitor=monitor_definition
)

# Schedule the monitoring job.
poller = ml_client.schedules.begin_create_or_update(model_monitor)
created_monitor = poller.result()